# Cobertura e qualidade do dump Argus

## Escopo e fuso horário

Os timestamps apresentados por este notebook estão em **UTC**. O BRT
(`America/Sao_Paulo`) deve ser aplicado apenas na visualização, nunca como
interpretação implícita dos dados de origem.

In [ ]:
from dump_explorer.charts import (
    cadence_histogram,
    collector_timeline,
    coverage_heatmap,
    missingness_bar,
)
from dump_explorer.config import load_settings
from dump_explorer.database import create_db_engine
from dump_explorer.quality import (
    load_collector_health,
    load_missingness,
    load_series_cadence,
    load_source_coverage,
    load_source_daily_coverage,
    load_table_coverage,
)

settings = load_settings()
engine = create_db_engine(settings)

## 1. coverage by table

Tabelas grandes usam estimativas; não há COUNT completo nelas.

In [ ]:
table_coverage = load_table_coverage(engine)
display(table_coverage)

## 2. Cobertura por signal_bucket.source

As épocas em milissegundos são convertidas para UTC.

In [ ]:
source_coverage = load_source_coverage(engine)
display(source_coverage)

## 3. source x day heatmap

Escolha uma janela finita em UTC para não executar varredura ampla.

In [ ]:
heatmap_start_ms = 0  # substitua por início UTC em epoch ms
heatmap_end_ms = 86_400_000  # substitua por fim UTC em epoch ms
source_day = load_source_daily_coverage(engine, heatmap_start_ms, heatmap_end_ms)
coverage_heatmap(source_day, x="day", y="source", value="rows").show()

## 4. Cadência para uma série selecionada

A série usa source, entity, metric e [start, end).

In [ ]:
series_source = "newrelic"
series_entity = "service-name"
series_metric = "error_rate_pct"
series_start_ms = 0  # substitua por início UTC em epoch ms
series_end_ms = 86_400_000  # substitua por fim UTC em epoch ms
cadence = load_series_cadence(
    engine, series_source, series_entity, series_metric, series_start_ms, series_end_ms
)
cadence_histogram(cadence).show()

## 5. Resumo de nulos e desconhecidos

Zero medido, nulo e ausência de bucket são estados distintos.

In [ ]:
small_table = "signal_bucket"
small_columns = ["source", "entity", "metric"]
missingness = load_missingness(engine, small_table, small_columns)
display(missingness)
missingness_bar(missingness).show()

## 6. Execuções dos coletores

Execuções `incomplete` não têm `finished_at`.

In [ ]:
collector_health = load_collector_health(engine
)
display(collector_health)
collector_timeline(collector_health).show()

## 7. Watermark e chegadas tardias

NPAW apresenta aproximadamente **28 horas** de atraso estrutural. Portanto,
o watermark deve preservar uma janela de releitura para chegadas tardias. A
ausência de tickets não é um rótulo negativo confiável: pode refletir falta de
abertura, coleta ou cobertura, e não ausência do incidente.